# S7.2 — Follow-up Handler

Context-aware follow-up Q&A with coreference resolution.
Detects follow-up queries, rewrites them using LLM + conversation history,
and re-retrieves from the knowledge base every time.

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.chat.follow_up import FollowUpHandler, FollowUpResult, is_follow_up, rewrite_query
from src.services.chat.memory import ChatMessage

print("\u2713 Imports successful")

✓ Imports successful


## FR-2: Follow-up Detection (Heuristic)

In [2]:
history = [
    ChatMessage(role="user", content="Explain transformer architecture"),
    ChatMessage(role="assistant", content="Transformers use self-attention mechanisms..."),
]

# Follow-up queries
assert is_follow_up("What about its limitations?", history) is True
assert is_follow_up("How about compared to BERT?", history) is True
assert is_follow_up("Why?", history) is True
assert is_follow_up("Can you elaborate on that?", history) is True

# Standalone queries
assert is_follow_up("Explain transformer architecture in detail", history) is False

# No history = never follow-up
assert is_follow_up("What about its limitations?", []) is False

print("\u2713 Follow-up detection works correctly")

✓ Follow-up detection works correctly


## FR-1: Query Rewriting with Coreference Resolution

In [3]:
from unittest.mock import AsyncMock, MagicMock
import asyncio

# Mock LLM for rewriting
mock_llm = AsyncMock()
mock_llm.generate.return_value = MagicMock(
    text="What are the limitations of the Transformer architecture?"
)

result = await rewrite_query("What about its limitations?", history, mock_llm)
assert result == "What are the limitations of the Transformer architecture?"
print(f"Rewritten: {result}")

# Empty history returns original
mock_llm2 = AsyncMock()
result2 = await rewrite_query("What about its limitations?", [], mock_llm2)
assert result2 == "What about its limitations?"
mock_llm2.generate.assert_not_called()

# LLM failure falls back to original
mock_llm3 = AsyncMock()
mock_llm3.generate.side_effect = RuntimeError("LLM down")
result3 = await rewrite_query("What about its limitations?", history, mock_llm3)
assert result3 == "What about its limitations?"

print("\u2713 Query rewriting works correctly")

LLM rewrite failed, falling back to original query
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/chat/follow_up.py", line 118, in rewrite_query
    response = await llm_provider.generate(prompt, temperature=0.0, max_tokens=256)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
RuntimeError: LLM down


Rewritten: What are the limitations of the Transformer architecture?
✓ Query rewriting works correctly


## FR-3: FollowUpHandler Orchestration

In [4]:
from src.services.rag.models import RAGResponse, SourceReference

# Mock dependencies
mock_memory = AsyncMock()
mock_memory.get_history.return_value = history

mock_llm = AsyncMock()
mock_llm.generate.return_value = MagicMock(
    text="What are the limitations of the Transformer architecture?"
)

mock_rag = AsyncMock()
mock_rag.aquery.return_value = RAGResponse(
    answer="Transformers have limitations including quadratic attention complexity [1].",
    sources=[SourceReference(
        index=1, arxiv_id="1706.03762", title="Attention Is All You Need",
        authors=["Vaswani"], arxiv_url="https://arxiv.org/abs/1706.03762",
    )],
    query="What are the limitations of the Transformer architecture?",
)

handler = FollowUpHandler(llm_provider=mock_llm, rag_chain=mock_rag, memory=mock_memory)
result = await handler.handle(session_id="demo", query="What about its limitations?")

assert result.is_follow_up is True
assert result.original_query == "What about its limitations?"
assert result.rewritten_query == "What are the limitations of the Transformer architecture?"
assert result.response is not None
assert "[1]" in result.response.answer
assert mock_memory.add_message.call_count == 2  # user + assistant stored

print(f"Original: {result.original_query}")
print(f"Rewritten: {result.rewritten_query}")
print(f"Answer: {result.response.answer}")
print(f"Is follow-up: {result.is_follow_up}")
print(f"Messages stored: {mock_memory.add_message.call_count}")
print("\u2713 FollowUpHandler orchestration works correctly")

Original: What about its limitations?
Rewritten: What are the limitations of the Transformer architecture?
Answer: Transformers have limitations including quadratic attention complexity [1].
Is follow-up: True
Messages stored: 2
✓ FollowUpHandler orchestration works correctly


## FollowUpResult Model

In [5]:
r = FollowUpResult(
    original_query="its limitations?",
    rewritten_query="What are the limitations of transformers?",
    is_follow_up=True,
    response=RAGResponse(answer="test", sources=[], query="test"),
)

assert r.original_query == "its limitations?"
assert r.rewritten_query == "What are the limitations of transformers?"
assert r.is_follow_up is True
assert r.response is not None

# Serializable
json_str = r.model_dump_json()
assert "original_query" in json_str

print("\u2713 FollowUpResult model works correctly")

✓ FollowUpResult model works correctly


## Graceful Degradation

In [6]:
# Without ConversationMemory
mock_rag2 = AsyncMock()
mock_rag2.aquery.return_value = RAGResponse(answer="test", sources=[], query="test")

handler_no_mem = FollowUpHandler(llm_provider=AsyncMock(), rag_chain=mock_rag2, memory=None)
result = await handler_no_mem.handle(session_id="s1", query="What about its limitations?")

# Treated as standalone (no history available)
assert result.is_follow_up is False
assert result.response is not None

print("\u2713 Graceful degradation without memory works correctly")

✓ Graceful degradation without memory works correctly
